# 10. Beyond a single image

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/10-data-cubes.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=10-data-cubes.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/10-data-cubes.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `6e24a78211a6351721ec15c5b034ad4c5e8d944938908b2377ea808d6ce57f21`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 10: Beyond a single image; abstract contour illustration](assets/10-data-cubes.svg)

# 10. Beyond a single image

Named dimensions and seasonal change

**Learning goals:** Use TerrIO with xarray; make a labelled time cube; distinguish anomalies from long-term trends.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=10-data-cubes.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Names protect meaning

An array with shape (12, 96, 96) is ambiguous until its dimensions are named. A cube also needs coordinate values, units, a CRS, a transform, and a missing-data policy. We first open a TerrIO GeoTIFF through its xarray integration.

In [ ]:
import xarray as xr
import terrio.xarray as txr
path = OUTPUT / "cube-base.tif"
raster().write(str(path))
base = txr.open_rasterio(str(path))
print("Dimensions:", base.dims, "shape:", base.shape)
print("Coordinates:", list(base.coords))
np.testing.assert_allclose(base.values.squeeze(), terrain())

## Construct a known seasonal signal

We invent twelve monthly temperatures with a sinusoidal seasonal cycle and a simple elevation effect. These are modelled teaching values, not weather observations or a climate projection. Month numbers carry no specific year.

In [ ]:
months = np.arange(1, 13)
z = terrain()
values = np.stack([15 + 8*np.sin(2*np.pi*(m-4)/12) - .0065*(z-1200) for m in months])
cube = xr.DataArray(values, dims=("month", "y", "x"),
                    coords={"month": months, "y": base.y.values, "x": base.x.values}, name="synthetic_temperature",
                    attrs={"units": "degC", "provenance": "Invented seasonal teaching model", "crs": CRS})
print(cube.shape, cube.dims)
assert cube.sizes["month"] == 12

## Separate the spatial and temporal questions

A spatial mean hides local variability. The central-pixel series and domain mean answer different questions, even when their curves have similar shapes in this deliberately simple model.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(months, cube.mean(("y", "x")), "o-", label="Domain mean")
ax.plot(months, cube.isel(y=48, x=48), "o-", label="Central pixel")
ax.set(xlabel="Synthetic month", ylabel="Temperature (°C)", title="An invented seasonal cycle")
ax.legend(); plt.show()

## An anomaly needs a reference

Subtracting this twelve-month mean gives departures from that reference. A single invented year cannot establish a climatic trend. Record the reference period whenever reporting an anomaly.

In [ ]:
anomaly = cube - cube.mean("month")
np.testing.assert_allclose(anomaly.mean("month"), 0, atol=1e-5)
image_grid([cube.sel(month=1).values, cube.sel(month=7).values], ["January-like model", "July-like model"], ["coolwarm", "coolwarm"], units="°C")
plt.show()
cube.mean(("y", "x")).to_dataframe().to_csv(OUTPUT / "synthetic-monthly-temperature.csv")
base.close()

## Your investigation

Add one missing month, compare skipna=True and skipna=False, and state what baseline your anomaly uses.

### Discussion guide

Skipping missing values changes the set of months supporting the mean. Report the months used and do not silently compare incompatible baselines.

## Sources and credit

- [Source 1](https://terrio.dev/xarray/)
- [Source 2](https://docs.xarray.dev/en/stable/user-guide/data-structures.html)
- [Source 3](https://terrio.dev/multidim/)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.